# Paper results 1/3: episode corpus and endpoint validity

**VLM-Initiated Behavior Tree Auditing of Robot VLA Execution (VIBTARVE).** The filename retains its legacy spelling. This is the first of three coordinated, independently runnable paper-results notebooks:

1. **This notebook:** evaluation corpus, simulator outcomes, episode-specific BT structure, and **Stage 1: endpoint validity**.
2. [Auditor sensitivity](VIBTARVE_Visualization.ipynb): **Stage 2**, including the selected Qwen3.8 and Luna 6 auditors.
3. [BT auditing results](VIBTARVE_Progress.ipynb): **Stages 3A--3E**, auditability, mission agreement, node progression, timing, and representative disagreements; optional continuous-progress supplement.

Results follow the paper's three-stage methodology. Interpretation accompanies each result. Tables retain exact counts and denominators; figures emphasize benchmark-task patterns. **Simulator goal-state success, human planned-path completion, and VLM judgments are separate signals.** Auditing is read-only with respect to robot execution. Selected Luna v5 contributes all 464 evaluation recordings, not its episode-1 tuning trials. Luna uses ten separate images per full window; historical Qwen3.8 uses eight-frame contact sheets, so these are not controlled input-matched comparisons.

See the [research overview](../docs/vibtarve.md), [dataset contract](../docs/btaudit2_dataset.md), and [analysis dependencies](../requirements-analysis.txt). The dataset is local and not distributed with Git. No GPU/server is required. All numeric data comes from the canonical [evaluation datasheet](../data/logs/btaudit2/btaudit2_episodes_2_21_visualization_rows.csv), selected by the frozen manifest. Refresh it after workbook changes with `python scripts/build_btaudit2_visualization_dataset.py --force`, then rerun the notebooks. Notebooks do not reread workbooks or rebuild raw data.

**Ordinal comparison:** auditors are compared as Node 1, Node 2, etc., retaining raw IDs and full model-specific node counts. Only shared positions are scored; extras are not comparable, not false incomplete. Unreached positions inside a known model contract remain incomplete. Whole-mission accuracy excludes unequal counts. This is positional agreement, not semantic equivalence of historical BTs. Some Qwen3 bindings use explicitly authorized frame-zero response-prefix inference with recorded full counts; the bundle's Ordinal node comparability table exposes this provenance. Synthetic completion milestones do not supply scoring judgments.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts').is_dir() and (p / 'README.md').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Start Jupyter from the repository or notebooks directory.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from scripts import paper_results as paper
from scripts import paper_bundle as bundle
plt.rcParams.update(bundle.STYLE)

DATASET_ROOT = REPO_ROOT / 'data/logs/btaudit2'
EPISODE_CSV = DATASET_ROOT / 'btaudit2_episodes_2_21_visualization_rows.csv'
COHORT_MANIFEST = REPO_ROOT / 'config/datasets/btaudit2_20261021_manifest.txt'
OUTPUT_DIR = bundle.DEFAULT_OUTPUT
results = paper.load_paper_results(EPISODE_CSV, cohort_manifest=COHORT_MANIFEST)
publication = paper.publication_tables(results)
pd.set_option('display.max_columns', 24)


## Evaluation corpus and simulator outcomes

**Paper table: evaluation corpus, simulator outcomes, and episode-specific BT structure.** Success-ever means the simulator predicate was true at any saved state; final-frame success is its value at the last saved state. A required execution node has a nonempty identifier, description, and success criterion. Counts come from episode-specific BTs, not a fixed template. Nodes/BT reports median [minimum--maximum].

Only episodes 2--21 contribute to this table. The 24 episode-1 prompt-tuning recordings are separate. A reviewed evaluation episode requires `complete=True` in the balanced schedule **and** a human Sheet1 row for every required node; neither workbook existence nor schedule membership alone suffices. Blank completion times on a confirmed reviewed node mean incomplete; missing human nodes are excluded, not counted as failures.

In [ ]:
display(publication['corpus'])
display(results.tables['review_coverage'])
total = results.tables['corpus'].iloc[-1]
display(Markdown(f"**Interpretation.** The evaluation cohort contains {int(total.episodes)} episodes and {int(total.required_nodes)} required nodes; {int(total.reviewed)} episodes have usable confirmed human review. Simulator success falls from {int(total.success_ever)} ever-successful episodes to {int(total.final_frame_success)} final-frame successes. These are endpoint predicates, not human planned-path or auditor completion rates."))


In [ ]:
fig_corpus = paper.plot_corpus(results)
plt.show()


## Stage 1: endpoint validity against human planned-path completion

Human mission completion requires **all required nodes** to have human completion times. Simulator success-ever and final-frame success are each evaluated as a predictor of that separate human signal. TP/FP/TN/FN use human completion as truth; unknown simulator states are explicitly excluded from metric denominators. Balanced accuracy is undefined when either human class is absent.

The three-way pattern table preserves all signals rather than silently replacing human labels with simulator outcomes. Any disagreement should be interpreted using the criteria and recorded evidence, not as automatic simulator or reviewer error.

In [ ]:
display(results.tables['endpoint_validity'].round(3))
display(results.tables['endpoint_patterns'])
endpoint_total = results.tables['endpoint_validity'].query("scenario == 'Total'")
for row in endpoint_total.itertuples():
    display(Markdown(f"**{row.simulator_signal}:** agrees with human mission completion in {row.tp + row.tn}/{row.n} known, reviewed pairs; {row.fp} simulator-only successes and {row.fn} human-only completions. Agreement does not make these signals interchangeable."))


In [ ]:
fig_endpoint = paper.plot_simulator_endpoint(results)
plt.show()
display(Markdown('**Figure denominator.** Task bars use known simulator outcomes; heatmap percentages are normalized within each human-completion row of the reviewed subset. Neither panel redefines human semantic completion.'))


## Representative execution context: three saved frames

Start, middle, and first simulator-success frames from the attached Cuttingboard-to-Pot example. Selection uses the datasheet's saved first-success step, time, and camera path; only the JPEG images are opened. Times are **simulator machine seconds**, using the saved time/step cadence, not human-review or replay-video seconds. A simulator-success image is not a human or auditor completion label. The exported frame-selection CSV makes this example reproducible.


In [ ]:
EXAMPLE_EPISODE = 'PosttrainPnPNovelFromCuttingboardToPotSplitA_GR1ArmsAndWaistFourierHands_Env_episode_3_of_21_20260616T035038Z'
fig_triptych, frame_selection = paper.plot_episode_triptych(results, EXAMPLE_EPISODE)
display(frame_selection)
plt.show()


## Export and reporting boundary

All three notebooks now export to [the consolidated paper bundle](../data/derived/vibtarve_paper/index.html): short-labeled PDF/300 dpi PNG figures, titled HTML/exact CSV tables, five compact captioned LaTeX tables, and a byte-for-byte full episode CSV copy. Figures use a 7.2-inch manuscript width and at least 9.5 pt text; use the PDFs at that width, not shrunk to one column. Each notebook refreshes its section; the index flags incomplete or mixed-source sections. Rebuild everything from the repository root with `python -m scripts.paper_bundle`. Change `OUTPUT_DIR` for a versioned snapshot. These are derived outputs, not source annotations; retain the manifest, review version, code, and environment with submitted results.

In [ ]:
bundle_index = bundle.export_stage(results, EPISODE_CSV, OUTPUT_DIR, '01', figures={'corpus': fig_corpus, 'endpoint': fig_endpoint})
display(Markdown(f'**Paper bundle:** [{bundle_index.name}]({bundle_index.as_uri()})'))
